### Importation des bibliothèques

In [7]:
import os
import scipy.io
import subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import normalized_mutual_info_score
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

### Configuration globale (Java + Palmetto + ETM + hyperparamètres)

In [8]:
JAVA_PATH = r"C:\Program Files\Java\jre1.8.0_51\bin\java.exe"
PALMETTO_JAR = r"C:\Users\Home\Documents\M2\Projet_PPD\palmetto\palmetto-0.1.0-jar-with-dependencies.jar"
WIKI_BD_DIR = r"C:\Users\Home\Documents\M2\Projet_PPD\palmetto\Wikipedia_bd\wikipedia_bd"
ETM_ROOT = r"C:\Users\Home\Documents\M2\Projet_PPD\output\ETM"
DATA_ROOT = r"C:\Users\Home\Documents\M2\Projet_PPD\data"

# Hyperparamètres ETM
SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
TOPN = 10
DATASETS = ["20NG", "IMDB", "YahooAnswer", "AGNews"]
K_VALUES = [50, 100]
lr = 0.002
batch_size = 1024
epochs = 100
rho_size = 300
t_hidden_size = 800
enc_drop = 0.5

print(f"ETM Configuration prête sur : {device}")
print(f"ETM_ROOT: {ETM_ROOT}")


ETM Configuration prête sur : cuda
ETM_ROOT: C:\Users\Home\Documents\M2\Projet_PPD\output\ETM


### Fonctions utilitaires ETM

In [9]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_dataset_etm(name):
    path = os.path.join(DATA_ROOT, name)
    train = sp.load_npz(os.path.join(path, "train_bow.npz")).toarray()
    test = sp.load_npz(os.path.join(path, "test_bow.npz")).toarray()
    vocab = [w.strip() for w in open(os.path.join(path, "vocab.txt"), encoding="utf-8")]
    test_labels = np.loadtxt(os.path.join(path, "test_labels.txt"), dtype=int)
    return train, test, vocab, test_labels

def topic_diversity(beta, vocab, topn=10):
    num_topics = beta.shape[0]
    list_w = []
    for k in range(num_topics):
        idx = beta[k].argsort()[-topn:][::-1]
        list_w.extend([vocab[i] for i in idx])
    td = len(set(list_w)) / len(list_w)
    return td

def purity_score(y_true, y_pred):
    y_voted_labels = np.zeros(y_true.shape)
    labels = np.unique(y_true)
    ordered_labels = np.arange(labels.shape[0])
    for i in range(len(labels)):
        y_true[y_true == labels[i]] = ordered_labels[i]
    labels = np.unique(y_true)
    bins = np.concatenate((labels, [np.max(labels)+1]), axis=0)
    for cluster in np.unique(y_pred):
        hist, _ = np.histogram(y_true[y_pred == cluster], bins=bins)
        winner = np.argmax(hist)
        y_voted_labels[y_pred == cluster] = winner
    return np.mean(y_voted_labels == y_true)

print("Fonctions utilitaires ETM prêtes")


Fonctions utilitaires ETM prêtes


### Architecture ETM

In [10]:
class ETM_Model(nn.Module):
    def __init__(self, vocab_size, num_topics, rho_size=300, t_hidden_size=800, dropout=0.5):
        super().__init__()
        self.num_topics = num_topics
        
        self.rho = nn.Linear(rho_size, vocab_size, bias=False)
        nn.init.orthogonal_(self.rho.weight)
        self.alpha = nn.Linear(rho_size, num_topics, bias=False)
        nn.init.orthogonal_(self.alpha.weight)
        
        self.encoder = nn.Sequential(
            nn.Linear(vocab_size, t_hidden_size),
            nn.BatchNorm1d(t_hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(t_hidden_size, t_hidden_size),
            nn.BatchNorm1d(t_hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.fc_mu = nn.Linear(t_hidden_size, num_topics)
        self.fc_logvar = nn.Linear(t_hidden_size, num_topics)
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
    
    def get_beta(self):
        logit_beta = self.alpha(self.rho.weight)
        return F.softmax(logit_beta, dim=0).t()
    
    def get_theta(self, x):
        self.eval()
        with torch.no_grad():
            x_norm = x / (x.sum(1, keepdim=True) + 1e-10)
            enc = self.encoder(x_norm)
            mu = self.fc_mu(enc)
            return F.softmax(mu, dim=-1).cpu().numpy()

print("Modèle ETM prêt")


Modèle ETM prêt


### Entraînement ETM 

In [11]:
def train_one_etm(dataset, K, seed=42):
    set_seed(seed)
    train_bow, test_bow, vocab, _ = load_dataset_etm(dataset)
    
    # Modèle ETM
    V = train_bow.shape[1]
    model = ETM_Model(V, K, rho_size, t_hidden_size, enc_drop).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1.2e-6)
    
    loader = DataLoader(TensorDataset(torch.from_numpy(train_bow).float()), 
                       batch_size=batch_size, shuffle=True)
    
    warmup = 40 if K == 50 else 60
    for epoch in tqdm(range(epochs), desc=f"ETM {dataset} K={K}", leave=False):
        model.train()
        kl_w = min(1.0, epoch / warmup)
        
        for batch in loader:
            x = batch[0].to(device)
            optimizer.zero_grad()
            recon_loss, kl_loss = model.forward_detailed(x)  # À adapter selon ton implémentation
            loss = recon_loss + kl_w * kl_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()
    
    # Sauvegarde ETM
    model.eval()
    beta = model.get_beta().detach().cpu().numpy()
    test_theta = model.get_theta(torch.from_numpy(test_bow).float().to(device))
    
    mat_path = rf"{ETM_ROOT}\{dataset}\{dataset}_ETM_K{K}_seed{seed}.mat"
    os.makedirs(os.path.dirname(mat_path), exist_ok=True)
    scipy.io.savemat(mat_path, {"beta": beta, "test_theta": test_theta})
    return mat_path

# ENTRAÎNEMENT 
print("Entraînement ETM ...")
missing = []
for dataset in DATASETS:
    for K in K_VALUES:
        path = rf"{ETM_ROOT}\{dataset}\{dataset}_ETM_K{K}_seed{SEED}.mat"
        if not os.path.exists(path):
            missing.append((dataset, K))

if missing:
    print(f"{len(missing)} modèles à entraîner")
    for dataset, K in missing:
        train_one_etm(dataset, K, SEED)
else:
    print("Tous les modèles ETM présents")

print("Entraînement terminé")


Entraînement ETM ...
Tous les modèles ETM présents
Entraînement terminé


### Calcul des métriques ETM (TD + Purity + NMI)

In [14]:
print("Calcul métriques ETM (TD + Purity + NMI)")
results = []

for dataset in tqdm(DATASETS, desc="ETM Metrics"):
    _, _, vocab, y_test = load_dataset_etm(dataset)
    
    for K in K_VALUES:
        mat_path = rf"{ETM_ROOT}\{dataset}\{dataset}_ETM_K{K}_seed{SEED}.mat"
        if not os.path.exists(mat_path):
            print(f"{mat_path}")
            continue
            
        data = scipy.io.loadmat(mat_path)
        beta = data["beta"]
        theta = data["test_theta"]
        
        td = topic_diversity(beta, vocab)
        preds = theta.argmax(1)
        pur = purity_score(y_test, preds)
        nmi = normalized_mutual_info_score(y_test, preds)
        
        results.append({
            'Dataset': dataset, 'K': K, 
            'TD': td, 'Purity': pur, 'NMI': nmi
        })
        print(f"{dataset} K={K} | TD={td:.3f} | Pur={pur:.3f} | NMI={nmi:.3f}")


df_etm.to_csv(f"{ETM_ROOT}/ETM_RESULTS.csv", index=False)
print(f"Sauvegarde faite : ETM_RESULTS.csv")


Calcul métriques ETM (TD + Purity + NMI)


ETM Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

20NG K=50 | TD=0.640 | Pur=0.535 | NMI=0.523
20NG K=100 | TD=0.284 | Pur=0.404 | NMI=0.490
IMDB K=50 | TD=0.494 | Pur=0.711 | NMI=0.078
IMDB K=100 | TD=0.138 | Pur=0.720 | NMI=0.111
YahooAnswer K=50 | TD=0.414 | Pur=0.560 | NMI=0.337
YahooAnswer K=100 | TD=0.151 | Pur=0.354 | NMI=0.229
AGNews K=50 | TD=0.270 | Pur=0.629 | NMI=0.381
AGNews K=100 | TD=0.104 | Pur=0.807 | NMI=0.519
Sauvegarde faite : ETM_RESULTS.csv


### Calcul de la métrique C_V Palmetto via Gensim

In [17]:
csv_path = f"{ETM_ROOT}/ETM_CV_PALMETTO.csv"
if os.path.exists(csv_path) and len(pd.read_csv(csv_path)) == len(DATASETS)*len(K_VALUES):
    print("C_V PALMETTO chargé")
    df_cv_palmetto = pd.read_csv(csv_path)
else:
    print("C_V PALMETTO via GENSIM")
    cv_palmetto_results = []

    for dataset in tqdm(DATASETS, desc="C_V Palmetto"):
        # 1. CHARGE DONNÉES
        train_bow, _, vocab_list, _ = load_dataset_etm(dataset)
        
        # 2. CORPUS pour C_V (requis)
        texts = []
        for doc in train_bow[:10000]: 
            doc_words = [vocab_list[i] for i, freq in enumerate(doc) if freq > 0]
            texts.append(doc_words)
        
        # 3. Dictionary + Corpus
        dictionary = Dictionary(texts)
        corpus = [dictionary.doc2bow(text) for text in texts]
        
        for K in K_VALUES:
            mat_path = rf"{ETM_ROOT}\\{dataset}\\{dataset}_ETM_K{K}_seed42.mat"
            if not os.path.exists(mat_path):
                print(f"Manquant: {mat_path}")
                continue
                
            # 4. Beta ETM → Topics
            data = scipy.io.loadmat(mat_path)
            beta = data["beta"]
            
            topics = []
            for k in range(min(K, beta.shape[0])):  
                idx = beta[k].argsort()[-TOPN:][::-1]
                topic_words = [vocab_list[i] for i in idx]
                topic_ids = [dictionary.token2id[word] for word in topic_words if word in dictionary.token2id]
                if len(topic_ids) >= 2:  
                    topics.append(topic_ids[:TOPN])
            
            if len(topics) == 0:
                print(f"Pas de topics valides {dataset} K={K}")
                continue
                
            # 5. C_V PALMETTO = Gensim 'c_v'
            cm = CoherenceModel(topics=topics, texts=texts, corpus=corpus, 
                              dictionary=dictionary, coherence='c_v')
            cv_score = cm.get_coherence()
            
            cv_palmetto_results.append({"Dataset": dataset, "K": K, "C_V_Palmetto": cv_score})
            print(f"{dataset:12} K={K:3d} | C_V={cv_score:.4f}")
    
    # Sauvegarde
    df_cv_palmetto = pd.DataFrame(cv_palmetto_results)
    df_cv_palmetto.to_csv(csv_path, index=False)

# AFFICHAGE FINAL 
print("\n C_V PALMETTO (résumé final)")
print(df_cv_palmetto.pivot(index="Dataset", columns="K", values="C_V_Palmetto").round(4))
print(" ETM_CV_PALMETTO.csv OK")


C_V PALMETTO chargé

 C_V PALMETTO (résumé final)
K               50      100
Dataset                    
20NG         0.4462  0.4011
AGNews       0.3782  0.3872
IMDB         0.3510  0.2865
YahooAnswer  0.6726  0.4286
 ETM_CV_PALMETTO.csv OK


### Comparaison ETM : Papier original vs Reproduction 

In [21]:
import pandas as pd

paper_etm = {
    "20NG":        {"K50": {"C_V": 0.375, "TD": 0.704, "Purity": 0.347, "NMI": 0.319},
                    "K100": {"C_V": 0.369, "TD": 0.573, "Purity": 0.394, "NMI": 0.339}},
    "IMDB":        {"K50": {"C_V": 0.346, "TD": 0.557, "Purity": 0.660, "NMI": 0.038},
                    "K100": {"C_V": 0.341, "TD": 0.371, "Purity": 0.648, "NMI": 0.037}},
    "YahooAnswer": {"K50": {"C_V": 0.354, "TD": 0.719, "Purity": 0.405, "NMI": 0.192},
                    "K100": {"C_V": 0.353, "TD": 0.624, "Purity": 0.428, "NMI": 0.208}},
    "AGNews":      {"K50": {"C_V": 0.364, "TD": 0.819, "Purity": 0.679, "NMI": 0.224},
                    "K100": {"C_V": 0.371, "TD": 0.773, "Purity": 0.674, "NMI": 0.204}},
}


# TES RÉSULTATS PALMETTO C_V (NOUVEAUX !)
reproduction_etm = {
    "20NG":        {"K50": {"C_V": 0.446 , "TD": 0.535, "Purity": 0.523, "NMI": 0.490},
                    "K100": {"C_V": 0.401, "TD": 0.404, "Purity": 0.490, "NMI": 0.284}},
    "IMDB":        {"K50": {"C_V": 0.351, "TD": 0.711, "Purity": 0.079, "NMI": 0.111},
                    "K100": {"C_V": 0.286, "TD": 0.720, "Purity": 0.111, "NMI": 0.494}},
    "YahooAnswer": {"K50": {"C_V": 0.673, "TD": 0.560, "Purity": 0.337, "NMI": 0.229},
                    "K100": {"C_V": 0.429, "TD": 0.354, "Purity": 0.229, "NMI": 0.414}},
    "AGNews":      {"K50": {"C_V": 0.378 , "TD": 0.629, "Purity": 0.381, "NMI": 0.520},
                    "K100": {"C_V": 0.387, "TD": 0.807, "Purity": 0.520, "NMI": 0.270}},
}

data = []
for dataset in ["20NG", "IMDB", "YahooAnswer", "AGNews"]:
    for k_str in ["K50", "K100"]:
        p, r = paper_etm[dataset][k_str], reproduction_etm[dataset][k_str]
        row = {"Dataset": dataset, "K": int(k_str[1:])}
        for m in ["C_V", "TD", "Purity", "NMI"]:
            row[f"{m}_Papier"] = p[m]
            row[f"{m}_Repro"]  = r[m] 
            row[f"{m}_Ecart"]  = round(r[m] - p[m], 3)
        data.append(row)

df = pd.DataFrame(data).set_index(["Dataset", "K"])


display(df.style
    .format("{:.3f}")
    .set_caption("ETM — C_V PALMETTO vs Papier Original")
)
